# Data Preprocessing 

This file shows how the NLP tools count vectorizer and doc2vec work with text in raw data. Considering the running time and the repeatability of the code, we use name and vec_size = 50 for the demonstration.


In [ ]:
import numpy as np
import pandas as pd

import pickle

x_train_original = pd.read_csv(r".\data\recipe_train.csv", index_col = False, delimiter = ',', header=0)
x_test_original = pd.read_csv(r".\data\recipe_test.csv", index_col = False, delimiter = ',', header=0)
train_corpus_name = x_train_original['name']
test_corpus_name = x_test_original['name']

## Count vectorizer

A count vectorizer converts documents to vectors which represent word counts. Each column in the output represents a different word and the values indicate the number of times that word appeared in the document. The overall size of a count vector matrix can be quite large (the number of columns is the total number of different words used across all documents in a corpus), but most entries in the matrix are zero (each document contains only a few of all the possible words). Therefore, it is most efficient to represent the count vectors as a sparse matrix.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vocab_name = CountVectorizer(stop_words='english').fit(train_corpus_name)

x_train_name = vocab_name.transform(train_corpus_name)
x_test_name = vocab_name.transform(test_corpus_name)

print(len(vocab_name.vocabulary_))

print(x_train_name.shape)
print(x_test_name.shape)

In [ ]:
with open('train_name_countvectorizer.pkl', 'wb') as f:
    pickle.dump(vocab_name, f)

np.savez('train_name_vec.npz', data=x_train_name)
np.savez('test_name_vec.npz', data=x_test_name)

## doc2vec

doc2vec methods are an extension of word2vec. word2vec maps words to a high-dimensional vector space in such a way that words which appear in similar contexts will be close together in the space. doc2vec does a similar embedding for multi-word passages. 

- This code may take a long time to run and require a good bit of memory

In [ ]:
import gensim

# size of the output vector (50 or 100)
vec_size = 50

# function to preprocess and tokenize text
def tokenize_corpus(txt, tokens_only=False):
    for i, line in enumerate(txt):
        tokens = gensim.utils.simple_preprocess(line)
        if tokens_only:
            yield tokens
        else:
            yield gensim.models.doc2vec.TaggedDocument(tokens, [i])

# tokenize a training corpus
corpus_name = list(tokenize_corpus(train_corpus_name))

model = gensim.models.doc2vec.Doc2Vec(vector_size=vec_size, min_count=2, epochs=40)
model.build_vocab(corpus_name)
model.train(corpus_name, total_examples=model.corpus_count, epochs=model.epochs)

doc = list(tokenize_corpus(train_corpus_name, tokens_only=True))

x_train_name = np.zeros((len(doc),vec_size))
for i in range(len(doc)):
    x_train_name[i,:] = model.infer_vector(doc[i])
    
print(x_train_name.shape)

In [ ]:
name_df = pd.DataFrame(x_train_name)
name_df.to_csv('train_name_doc2vec50.csv', index=False, header=False)